In [ ]:
# NOTEBOOK PARAMETERS
selected_model = 'neptune_1'
ensemble_size = 8
years = [2019,2020,2021]

# ENSO Forecast Composite: Emulator vs Reanalysis

This notebook provides a comprehensive analysis of ENSO (El Niño Southern Oscillation) events over a yearly forecast horizon, comparing climate emulator outputs with ocean reanalysis data.

Main steps and features include:

- **Data loading and alignment**: Opens forecast and reanalysis datasets, aligns their time dimensions, and prepares them for analysis.

- **Temporal aggregation**: Resamples data to monthly means and selects surface sea temperature (SST) fields for direct comparison.

- **Climatology and anomaly computation**: Calculates climatology and SST anomalies for both forecast and reanalysis datasets.

- **ENSO index calculation**: Computes Oceanic Niño Indices (ONI) for multiple Niño regions, enabling direct comparison between emulator and reanalysis time series.

- **Event detection**: Identifies El Niño and La Niña years based on SST anomalies.

- **Spatial composite analysis**: Computes and visualizes spatial composites of SST anomalies during El Niño and La Niña events for both forecast and reanalysis.

- **Visualization**: Plots time series and spatial maps to assess emulator skill against reanalysis, with options to save figures for further use.

The notebook is designed to evaluate the performance and consistency of climate emulators in reproducing ENSO-related SST anomalies, providing both temporal and spatial insights over a 10-year forecast period.

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import xarray as xr
import sys
import os

sys.path.append('library')
from atmocean.enso import (
    compute_climatology,
    compute_ocean_nino_index,
    compute_bias_sst,
    NINO_COORD_REGIONS,
    SEASONS,
    monthly_resample_with_prediction_timedelta,
    monthly_groupby_with_prediction_timedelta,
)
from atmocean.visualize import get_sst_bias_colormap, get_colormap_from_cmap
from atmocean.utils import (
    clean_forecasts,
    remove_ocean_variables_suffix,
    align_prediction_timedelta,
)
os.system('hostname')
%load_ext autoreload
%autoreload 2

In [ ]:
start_date = '01-01'
end_date = '12-31'
lead_times = [0, 10, 20, 30, 40, 50, 60]

main_dir = 'PATH-TO-MAIN-DIRECTORY'
data_dir = os.path.join(main_dir, 'data')
forecast_dir = os.path.join(data_dir, 'forecasts')

if 'highres' not in selected_model:
    climatology_src = os.path.join(forecast_dir, f'climatology-from_1993_to_2017.zarr')
else:
    climatology_src = os.path.join(forecast_dir, f'climatology_025-from_1993_to_2017.zarr')

if ensemble_size is None:
    forecasts_srcs = [
        os.path.join(forecast_dir, f'{selected_model}-from_{year}-{start_date}_to_{year}-{end_date}.zarr')
        for year in years
    ]
    out_img_dir = os.path.join(main_dir, f'validation/img/{selected_model}/s2s-{years[0]}-{years[-1]}/ocean_indices/enso')
else:
    forecasts_srcs = [
        os.path.join(forecast_dir, f'{selected_model}-from_{year}-{start_date}_to_{year}-{end_date}-ENS-{ensemble_size:02d}.zarr')
        for year in years
    ]
    out_img_dir = os.path.join(main_dir, f'validation/img/{selected_model}/s2s-{years[0]}-{years[-1]}-ENS-{ensemble_size:02d}/ocean_indices/enso')

os.makedirs(out_img_dir, exist_ok=True)
plot_kwargs = dict(dpi=100, bbox_inches='tight')
save_figures = True

In [ ]:
from dask.distributed import Client, LocalCluster, progress, wait
from tqdm import tqdm
import numpy as np
import dask
import ctypes

# spill to disk before hitting hard memory limits (avoids KilledWorker)
dask.config.set({
    'distributed.worker.memory.target': 0.6,
    'distributed.worker.memory.spill': 0.7,
    'distributed.worker.memory.pause': 0.8,
    'distributed.worker.memory.terminate': 0.95,
})

n_workers = 4
memory_limit = 948 // n_workers
cluster = LocalCluster(
    n_workers=n_workers, threads_per_worker=1,
    memory_limit=f'{memory_limit}GB',
    local_directory=os.path.join(main_dir, 'tmp_dask'),  # spill directory
)
client = Client(cluster)

def _trim_memory() -> int:
    libc = ctypes.CDLL("libc.so.6")
    return libc.malloc_trim(0)

client.run(_trim_memory)
client

In [ ]:
# open forecasts
forecasts = xr.open_mfdataset(
    forecasts_srcs, chunks={}, parallel=True, engine='zarr', preprocess=clean_forecasts,
    combine='nested', concat_dim='time', data_vars='minimal', coords='minimal', compat='override',
)
# manipulate the time dimension to align
if '_025' not in selected_model:
    forecasts = forecasts.isel(prediction_timedelta=lead_times)
forecasts = forecasts.isel(depth=0, level=0, missing_dims='ignore')
# remove ocean variables suffix (oras, cglo, glor)
forecasts = remove_ocean_variables_suffix(forecasts)

In [ ]:
# open historical climatology
climatology = xr.open_zarr(climatology_src, chunks={})
if 'highres' not in selected_model:
    climatology = climatology.isel(prediction_timedelta=lead_times)
climatology = climatology.isel(depth=0, level=0, missing_dims='ignore')
# remove ocean variables suffix (oras, cglo, glor)
climatology = remove_ocean_variables_suffix(climatology)

In [ ]:
if 'highres' in selected_model:
    forecasts['longitude'] = climatology.longitude.values

In [ ]:
# select only SST
forecasts = forecasts['thetao']
climatology = climatology['thetao']
# resample over time dimension
forecasts = monthly_resample_with_prediction_timedelta(forecasts)
climatology = monthly_groupby_with_prediction_timedelta(climatology)

In [ ]:
forecasts, climatology = client.compute([forecasts, climatology])
progress([forecasts, climatology])
wait([forecasts, climatology])

In [ ]:
forecasts = forecasts.result().load() if hasattr(forecasts, 'result') else forecasts.load()
climatology = climatology.result().load() if hasattr(climatology, 'result') else climatology.load()

In [ ]:
# get the nino region
lat1, lat2, lon1, lon2 = NINO_COORD_REGIONS['3.4']
region = NINO_COORD_REGIONS['3.4']
nino_index_by_lead_time = {}
# compute nino index for each lead time
for l,lead_time in enumerate(tqdm(lead_times)):
    # get the dataset for this lead time
    if ensemble_size is None:
        ds = forecasts.isel(prediction_timedelta=l, missing_dims='ignore')
    else:
        ds = forecasts.isel(prediction_timedelta=l, missing_dims='ignore').mean(dim='ensemble_member', skipna=True)
    # ds = forecasts.isel(prediction_timedelta=l, missing_dims='ignore')
    clm = climatology.isel(prediction_timedelta=l, missing_dims='ignore')
    # use dask delayed to delay the computation of the ocean nino index
    nino_index_by_lead_time[f'{lead_time}'] = compute_ocean_nino_index(ds, clm, region=region)

In [ ]:
# compute the delayed work
nino_index_by_lead_time = dask.compute(nino_index_by_lead_time)
nino_index_by_lead_time = nino_index_by_lead_time[0]

In [ ]:
season = 'all'

In [ ]:
forecast_bias_sst = {}
for lead_time in lead_times:
    anom_ds, ocean_nino_index, ocean_nino_index_bool, elnino_years, lanina_years = nino_index_by_lead_time[f'{lead_time}']
    forecast_bias_sst[f'{lead_time}'] = compute_bias_sst(anom_ds, elnino_years, lanina_years, SEASONS[season])

In [ ]:
from matplotlib.colors import BoundaryNorm, ListedColormap
from matplotlib.ticker import ScalarFormatter
from matplotlib.gridspec import GridSpec
from matplotlib import pyplot as plt
import matplotlib.ticker as mticker
import cartopy.feature as cfeature
import cartopy.crs as ccrs

In [ ]:
grid_list = [
    'gs[0,0:2]',
    'gs[0,2:4]',
    'gs[0,4:6]',
    'gs[1,0:2]',
    'gs[1,2:4]',
    'gs[1,4:6]',
    'gs[2,2:4]',
]

In [ ]:
central_longitude = 180
cmap, norm, levels = get_sst_bias_colormap(-3.0, 3.0)

lats, lons = forecasts.latitude.data, forecasts.longitude.data
fig, axes = plt.subplots(1, 1, figsize=(12,5.5))
axes.axis('off')

# initialize grid spec
gs = GridSpec(3, 6, figure=fig)
axs = []

period = season if season != 'all' else 'Annual'
fig.suptitle(f'El Niño - La Niña Spatial Composite SST ({period})', fontsize=16, y=1.05)

for i, lead_time in enumerate(lead_times):
    ax = fig.add_subplot(eval(grid_list[i]), projection=ccrs.Robinson(central_longitude=central_longitude))
    bias = forecast_bias_sst[f'{lead_time}']
    im = ax.contourf(lons, lats, bias, levels=levels, cmap=cmap, norm=norm, extend='both', transform=ccrs.PlateCarree())
    title = 'Reanalysis' if lead_time == 0 else f"Forecast t + {lead_time}"
    ax.set_title(title, fontsize=14)
    # set white background for spatial plots
    ax.set_facecolor("white")
    # plot the coastlines along the image
    ax.coastlines(resolution='110m', linewidth=0.6, edgecolor="black", alpha=1.0)
    # color the land
    ax.add_feature(cfeature.LAND, edgecolor="black", facecolor='#B1B2B4', alpha=0.3, zorder=1)
    # create ticks every 90 deg
    longitude_ticks = mticker.MultipleLocator(base=90.0)
    gl = ax.gridlines(draw_labels=True, xlocs=longitude_ticks, linestyle="--", linewidth=0.5, color="gray", alpha=0.5)
    gl.top_labels = False
    gl.right_labels = False
    gl.xlabel_style = {"size": 12}
    gl.ylabel_style = {"size": 12}
    axs.append(ax)

# single colorbar
cax = fig.add_axes([0.25, -0.05, 0.5, 0.03]) # [left, bottom, width, height]
cbar = fig.colorbar(
    im, ax=axs, cax=cax,
    orientation='horizontal', pad=0.05,
    aspect=60, shrink=0.9, label='°C', 
    format=ScalarFormatter()
)
plt.subplots_adjust(left=0.05, right=0.95, bottom=0.05, top=0.95, wspace=-0.5, hspace=0.4)
outfile = os.path.join(out_img_dir, 'elnino_lanina_spatial_composite.png') if save_figures else None
plt.savefig(outfile, **plot_kwargs) if outfile else plt.show()

In [ ]:
central_longitude = 180
cmap, norm, levels = get_colormap_from_cmap('RdBu_r', vmin=-1.0, vmax=1.0) # get_sst_bias_colormap(-1.0, 1.0)

lats, lons = forecasts.latitude.data, forecasts.longitude.data
fig, axes = plt.subplots(1, 1, figsize=(15,5.5))
axes.axis('off')

# initialize grid spec
gs = GridSpec(2, 6, figure=fig)
axs = []

period = season if season != 'all' else 'Annual'
fig.suptitle(f'El Niño - La Niña Spatial Composite Bias SST ({period})', fontsize=16, y=1.05)

for i, lead_time in enumerate(lead_times[1:]):
    ax = fig.add_subplot(eval(grid_list[i]), projection=ccrs.Robinson(central_longitude=central_longitude))
    bias = forecast_bias_sst[f'{lead_time}'] - forecast_bias_sst[f'0']
    im = ax.contourf(lons, lats, bias, cmap=cmap, norm=norm, levels=levels, extend='both', transform=ccrs.PlateCarree())
    title = f"Forecast t + {lead_time}"
    ax.set_title(title, fontsize=14)
    # set white background for spatial plots
    ax.set_facecolor("white")
    # plot the coastlines along the image
    ax.coastlines(resolution='110m', linewidth=0.6, edgecolor="black", alpha=1.0)
    # color the land
    ax.add_feature(cfeature.LAND, edgecolor="black", facecolor='#B1B2B4', alpha=0.3, zorder=1)
    # create ticks every 90 deg
    longitude_ticks = mticker.MultipleLocator(base=90.0)
    gl = ax.gridlines(draw_labels=True, xlocs=longitude_ticks, linestyle="--", linewidth=0.5, color="gray", alpha=0.5)
    gl.top_labels = False
    gl.right_labels = False
    gl.xlabel_style = {"size": 12}
    gl.ylabel_style = {"size": 12}
    axs.append(ax)

# single colorbar
cax = fig.add_axes([0.25, -0.05, 0.5, 0.03]) # [left, bottom, width, height]
cbar = fig.colorbar(
    im, ax=axs, cax=cax,
    orientation='horizontal', pad=0.05,
    aspect=60, shrink=0.9, label='°C', 
    format=ScalarFormatter()
)
plt.subplots_adjust(left=0.05, right=0.95, bottom=0.05, top=0.95, wspace=0.0, hspace=0.4)
outfile = os.path.join(out_img_dir, 'elnino_lanina_spatial_composite_bias.png') if save_figures else None
plt.savefig(outfile, **plot_kwargs) if outfile else plt.show()

In [ ]:
# fig, ax = plt.subplots(figsize=(20,4))
# plt.title(f'El Niño SST Anomalies Time-series in the 3.4 Region', fontdict={'weight':'bold', 'size':14})
# alpha = 1.0
# for lead_time in lead_times:
#     _, fc_oni, _, _, _ = nino_index_by_lead_time[f'{lead_time}']
#     label = 'Reanalysis' if lead_time == 0 else f"Forecast t + {lead_time}"
#     kwargs = dict(label=label, color='black', alpha=alpha)
#     if lead_time != 0:
#         kwargs['linestyle'] = 'dashed'
#     ax.plot(fc_oni.time, fc_oni, **kwargs)
#     ax.hlines(0, forecasts.time[0], forecasts.time[-1], color='black', alpha=0.6, zorder=-999, linewidth=0.5)
#     ax.hlines(0.5, forecasts.time[0], forecasts.time[-1], color='black', alpha=0.6, zorder=-999, linewidth=0.5)
#     ax.hlines(-0.5, forecasts.time[0], forecasts.time[-1], color='black', alpha=0.6, zorder=-999, linewidth=0.5)
#     alpha -= 0.1

#     ax.fill_between(fc_oni.time, fc_oni, 0.5, where=fc_oni>=0.5, alpha=0.5, color='tab:red')
#     ax.fill_between(fc_oni.time, fc_oni, -0.5, where=fc_oni<=-0.5, alpha=0.5, color='tab:red')

# ax.set_xlim([
#     forecasts.time.values[0],
#     forecasts.time.values[-1],
# ])

# ticks = [d for d in pd.to_datetime(forecasts.time.values) if d.day==1]
# ticklabels = [f'{d.year}-{d.month}' for d in pd.to_datetime(forecasts.time.values) if d.day==1]

# ax.set_xticks(ticks)
# ax.set_xticklabels(ticklabels, rotation=45)

# ax.set_ylabel('SST Anomaly', fontdict={'weight':'bold', 'size':12})
# ax.set_xlabel('Time', fontdict={'weight':'bold', 'size':12})
# ax.legend()
# outfile = os.path.join(out_img_dir, f'el_nino_index_anomaly_on_3.4.png')
# plt.savefig(outfile, **plot_kwargs) if save_figures else plt.show()

In [ ]:
from atmocean.visualize import plot_dmi_timeseries, discrete_cmap

colors = discrete_cmap(len(lead_times)+2, 'Blues_r')
colors = [colors(i) for i in range(len(lead_times))]

seasonal_colors = {
    f'{lead_time}': colors[i]
    for i, lead_time in enumerate(lead_times)
}
seasonal_names = {
    f'{lead_time}': 'Reanalysis' if lead_time == 0 else f"Forecast t + {lead_time}"
    for lead_time in lead_times
}
models_data = {}
for lt in lead_times:
    _, fc_oni, _, _, _ = nino_index_by_lead_time[f'{lt}']
    models_data[f'{lt}'] = fc_oni

outfile = os.path.join(out_img_dir, f'el_nino_index_anomaly_on_3.4.png') if save_figures else None
plot_dmi_timeseries(
    data=models_data,
    colors=seasonal_colors,
    names=seasonal_names,
    correlation='spearman',
    dmi='El Niño SST Anomalies Time-series in the 3.4 Region',
    threshold=0.5,
    dmi_label=False,
    outfile=outfile,
)

In [ ]:
from atmocean.visualize import plot_map_on_grid

lead_times_figure = [0,10,30,60] # lead_times_figure = lead_times
grid_list_top = ['gs[0,0:2]','gs[0,2:4]','gs[0,4:6]','gs[0,6:8]']
grid_list_bottom = ['gs[1,0:2]','gs[1,2:4]','gs[1,4:6]','gs[1,6:8]']

cmap_top, norm_top, levels_top = get_sst_bias_colormap(-3, 3)
cmap_bottom, norm_bottom, levels_bottom = get_colormap_from_cmap('RdBu_r', -1.0, 1.0)

enso_list = []
enso_bias_list = []
for i,lt in enumerate(lead_times_figure):
    enso_list.append(
        forecast_bias_sst[f'{lt}']
    )
    enso_bias_list.append(
        forecast_bias_sst[f'{lt}'] - forecast_bias_sst[f'0']
    )

central_longitude = 180
lats, lons = enso_list[0].latitude.data, enso_list[0].longitude.data

fig, axes = plt.subplots(1, 1, figsize=(12,3))

axes.axis('off')

# initialize grid spec
gs = GridSpec(2, 8, figure=fig)
axs_top, axs_bottom = [], []
eps = 0.0 # 1e-2

for i,lt in enumerate(lead_times_figure):
    im_top, ax = plot_map_on_grid(
        bias=enso_list[i], grid=eval(grid_list_top[i]), title='Ground Truth' if i == 0 else f't+{lt}', title_kwargs=dict(fontsize=11), # f't+{i}ΔT', title_kwargs=dict(fontsize=11), 
        fig=fig, lons=lons, lats=lats, central_longitude=central_longitude, cmap=cmap_top, norm=norm_top, levels=levels_top
    )
    axs_top.append(ax)

for i,lt in enumerate(lead_times_figure):
    if i == 0: 
        ax = fig.add_subplot(eval(grid_list_bottom[0]))
        ax.axis('off')
        continue
    im_bottom, ax = plot_map_on_grid(
        bias=enso_bias_list[i], grid=eval(grid_list_bottom[i]), title='', title_kwargs=dict(fontsize=11), # ='Ground Truth' if i == 0 else f't+{lt}',
        fig=fig, lons=lons, lats=lats, central_longitude=central_longitude, cmap=cmap_bottom, norm=norm_bottom, levels=levels_bottom
    )
    axs_bottom.append(ax)

# single colorbar
cax_top = fig.add_axes([0.95, 0.55, 0.01, 0.4]) # [left, bottom, width, height]
cax_bottom = fig.add_axes([0.95, 0.05, 0.01, 0.4]) # [left, bottom, width, height]
cbar_top = fig.colorbar(
    im_top, ax=axs_top, cax=cax_top, orientation='vertical', label='°C',
)
cbar_bottom = fig.colorbar(
    im_bottom, ax=axs_bottom, cax=cax_bottom, orientation='vertical', label='°C',
)
# adjust the plot
plt.subplots_adjust(
    left=0.05, right=0.95, bottom=0.05, top=0.95, wspace=0.0, hspace=0.3
)

# save or plot figure
outfile = os.path.join(out_img_dir, f'el_nino_southern_oscillation.pdf')
plt.savefig(outfile, **plot_kwargs) if outfile else plt.show()